# M1. 서울 따릉이 대여 패턴

서울시가 공개한 따릉이 대여 이력 사흘치(42만 건)를 pandas 로 정제하고 DuckDB SQL 로 시간대 · 요일 · 대여소별 패턴을 찾습니다.

- 단계: 기초 미니 프로젝트 · 정형 · 예상 3~4시간
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 서울 열린데이터광장 — 서울특별시 공공자전거 대여이력 정보(OA-15182) 2026년 6월분 중 6/5(금)~6/7(일), 공공누리 1유형(출처표시)

**빈칸 노트북** — 코드 속 `___①___` 자리 12곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/M1/M1_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

따릉이는 하루 10만 건 넘게 빌려집니다. 서울시는 그 대여 기록을 한 달에 한 파일씩 공개합니다.
이 프로젝트에서는 그중 금 · 토 · 일 사흘치를 받아 **「언제, 어디서 가장 많이 빌리나」**를 숫자로 답합니다.

흐름은 1~3주차에 배운 그대로입니다.

```text
CSV(원본, CP949) → pandas 읽기 · 정제 → DuckDB SQL 집계 → Parquet 저장 · 그래프
```

다 끝내면 다음 세 가지를 말할 수 있습니다.

- 가장 붐비는 시간대와, 평일과 휴일에 그 시간이 어떻게 다른지
- 가장 많이 빌려 간 대여소 10곳
- 「빌렸다가 바로 반납한」 기록을 왜 빼야 하는지

## 1단계 · 준비 — 라이브러리와 데이터

Colab 에는 pandas 와 DuckDB 가 이미 깔려 있습니다. 데이터는 공개 자료실에서 받습니다.

원본은 서울 열린데이터광장의 「서울특별시 공공자전거 대여이력 정보」 2026년 6월 파일(418만 건, 약 730MB)입니다.
실습 시간을 줄이려고 6월 5일(금)부터 7일(일)까지 사흘치만 잘라 두었습니다. 열과 글자 인코딩은 원본 그대로입니다.

In [ ]:
import os, shutil, urllib.request
import pandas as pd
import duckdb

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

path = fetch("seoul_bike_20260605_07.csv.gz")
print(path, round(os.path.getsize(path) / 1e6, 1), "MB")

> **확인 포인트** — `data/seoul_bike_20260605_07.csv.gz 18.9 MB` 가 찍힙니다.

> **Aebon's Tip** — 한 달치 전체를 직접 받아 보고 싶으면 서울 열린데이터광장에서 「공공자전거 대여이력」을 검색해 내려받으세요.
> 압축하지 않은 CSV 라 `pd.read_csv` 에 파일 이름만 바꿔 넣으면 같은 코드가 그대로 돕니다(시간은 열 배쯤 걸립니다).

## 2단계 · 읽기 — 일부러 한 번 실패하기

공공기관 CSV 는 아직 **CP949**(한글 윈도 인코딩)로 저장된 것이 많습니다. pandas 는 기본으로 UTF-8 로 읽기 때문에 그냥 읽으면 실패합니다.
먼저 실패를 보고, 인코딩을 알려 주어 다시 읽습니다.

In [ ]:
try:
    pd.read_csv(path, nrows=5)
except UnicodeDecodeError as e:
    print("읽기 실패:", str(e)[:80])

In [ ]:
raw = pd.read_csv(path, encoding="___①___", dtype=___②___)   # ① 한글 윈도 인코딩 이름 · ② 모든 열을 글자로
print(raw.shape)
raw.head(3)

`dtype=str` 로 모든 열을 글자로 읽었습니다. 대여소 번호 `00502` 의 앞자리 0이 숫자로 읽히면 `502` 가 되어 다른 번호와 섞이기 때문입니다.
숫자와 날짜는 다음 단계에서 열마다 골라 바꿉니다.

> **확인 포인트** — `(427258, 16)` — 42만 7천여 행, 16개 열입니다.

In [ ]:
assert raw.shape == (427258, 16)

## 3단계 · 정제 — 이름 바꾸고, 형 바꾸고, 이상한 기록 빼기

열 이름이 한글에 띄어쓰기까지 있어서 SQL 에 쓰기 불편합니다. 쓸 열만 골라 영어 이름으로 바꿉니다.

In [ ]:
cols = {
    "자전거번호": "bike_id",
    "대여일시": "rent_at",
    "대여 대여소번호": "rent_station",
    "대여 대여소명": "rent_station_name",
    "반납일시": "return_at",
    "반납대여소번호": "return_station",
    "이용시간(분)": "use_min",
    "이용거리(M)": "use_m",
}
df = raw.___③___(columns=cols)[list(cols.values())]   # ③ 열 이름 바꾸기

df["rent_at"] = pd.___④___(df["rent_at"])   # ④ 글자를 날짜로
df["return_at"] = pd.to_datetime(df["return_at"])
df["use_min"] = pd.to_numeric(df["use_min"], errors="___⑤___")   # ⑤ 못 바꾸면 NaN 으로
df["use_m"] = pd.to_numeric(df["use_m"], errors="coerce")
df.dtypes

이제 빈 값을 셉니다.

In [ ]:
df.isna().sum()

반납 대여소 번호가 빈 행이 2,060개 있습니다. 반납 시각은 모두 있으니 대여 기록으로는 살려 두고, 반납 대여소를 쓰는 분석에서만 뺍니다.

다음은 **「빌렸다가 바로 반납한」 기록**입니다. 이용 시간이 1분 이하이고 빌린 곳에 그대로 반납했다면, 실제로 타고 다닌 것이 아니라 잘못 빌렸거나 자전거 상태를 보고 돌려놓은 경우일 가능성이 큽니다.
이런 기록이 섞이면 「많이 빌린 대여소」 순위가 부풀려집니다.

In [ ]:
quick = (df["use_min"] <= 1) & (df["rent_station"] == df["___⑥___"])   # ⑥ 반납 대여소 열
print("바로 반납:", quick.sum())

clean = df[___⑦___].copy()   # ⑦ 조건을 뒤집어 남길 행 고르기
print("정제 후:", len(clean))

> **확인 포인트** — 바로 반납 13,046건을 빼고 **414,212건**이 남습니다.

In [ ]:
assert quick.sum() == 13046 and len(clean) == 414212

> **자주 겪는 오류** — `df[quick == False]` 처럼 써도 결과는 같지만, 조건 뒤집기는 `~` 가 표준입니다.
> `not quick` 은 표 전체를 하나의 참/거짓으로 바꾸려다 `ValueError: The truth value of a Series is ambiguous` 가 납니다.

## 4단계 · DuckDB 로 시간대 · 요일 집계

DuckDB 는 같은 파이썬 안에 있는 pandas 표 이름을 그대로 SQL 의 테이블처럼 씁니다. 따로 적재할 필요가 없습니다.

In [ ]:
by_hour = duckdb.sql("""
    SELECT ___⑧___(rent_at) AS h, count(*) AS n   -- ⑧ 시각에서 시만 꺼내는 함수
    FROM clean
    GROUP BY h
    ORDER BY n DESC
    LIMIT 5
""").df()
by_hour

> **확인 포인트** — 1위는 **18시 35,888건**, 2위 17시, 3위 19시입니다 — 퇴근 시간입니다.

In [ ]:
assert by_hour.iloc[0]["h"] == 18 and by_hour.iloc[0]["n"] == 35888

이번에는 날짜마다 가장 붐빈 시간을 하나씩 뽑습니다. 「그룹마다 1등」은 윈도 함수 `row_number()` 로 풉니다(2주차 8장).

In [ ]:
peak = duckdb.sql("""
    SELECT d, h, n FROM (
        SELECT strftime(rent_at, '%m-%d') AS d, hour(rent_at) AS h, count(*) AS n,
               row_number() OVER (PARTITION BY ___⑨___ ORDER BY count(*) DESC) AS r   -- ⑨ 날짜마다 따로 순위
        FROM clean
        GROUP BY d, h
    )
    WHERE r = ___⑩___   -- ⑩ 1등만
    ORDER BY d
""").df()
peak

> **확인 포인트** — 6/5(금) 18시 16,833건 · 6/6(토) 18시 10,117건 · 6/7(일) 17시 9,337건.
> 금요일은 출퇴근 정점이 뚜렷하고, 6/6(현충일) · 6/7은 정점이 낮고 이릅니다.

In [ ]:
assert peak["n"].tolist() == [16833, 10117, 9337]

## 5단계 · 많이 빌린 대여소 10곳

In [ ]:
top10 = duckdb.sql("""
    SELECT rent_station, any_value(rent_station_name) AS name, count(*) AS n
    FROM clean
    GROUP BY ___⑪___   -- ⑪ 대여소마다
    ORDER BY n DESC
    LIMIT 10
""").df()
top10

> **확인 포인트** — 1위는 **한강버스 망원 선착장(05515) 2,235건**, 2위 올림픽공원역 3번출구 2,006건입니다.
> 주말이 낀 사흘이라 한강 근처 대여소가 위에 많습니다.

In [ ]:
assert top10.iloc[0]["rent_station"] == "05515" and top10.iloc[0]["n"] == 2235

> **Aebon's Tip** — 3단계에서 「바로 반납」을 빼지 않은 `df` 로 같은 SQL 을 돌려 보세요. 순위가 어떻게 달라지는지 보면 정제가 왜 필요한지 바로 보입니다.

## 6단계 · 저장과 그래프

정제한 표는 Parquet 으로 저장합니다. 열마다 형이 함께 저장되고 크기도 CSV 의 몇 분의 일입니다(3주차 13장).

In [ ]:
clean.___⑫___("data/bike_clean.parquet", index=False)   # ⑫ Parquet 으로 저장
print(round(os.path.getsize("data/bike_clean.parquet") / 1e6, 1), "MB")

back = duckdb.sql("SELECT count(*) AS n FROM 'data/bike_clean.parquet'").fetchall()[0][0]
print("다시 읽은 행 수:", back)
assert back == 414212

In [ ]:
import matplotlib.pyplot as plt

hourly = duckdb.sql("""
    SELECT strftime(rent_at, '%m-%d') AS d, hour(rent_at) AS h, count(*) AS n
    FROM clean GROUP BY d, h ORDER BY d, h
""").df().pivot(index="h", columns="d", values="n")

hourly.plot(figsize=(9, 4), marker="o")
plt.title("Seoul bike rentals by hour (2026-06-05 ~ 06-07)")
plt.xlabel("hour")
plt.ylabel("rentals")
plt.xticks(range(0, 24, 2))
plt.grid(alpha=.3)
plt.show()

> **확인 포인트** — 금요일 선은 아침 8시와 저녁 18시에 두 번 솟고, 토 · 일 선은 오후에 넓게 퍼집니다.

## 7단계 · 더 해 보기

- 반납 대여소까지 써서 **많이 오간 구간(대여소 → 반납 대여소) 10개**를 뽑아 보세요. 반납 대여소가 빈 2,060행은 빼고 셉니다.
- 이용 시간이 240분을 넘는 기록은 몇 건인가요? 정제에서 뺄지 남길지 이유와 함께 정해 보세요.
- 한 달치 파일을 받아 날짜별 대여량을 그리면, P2(날씨 × 따릉이)의 출발점이 됩니다.

## 정답

① `cp949` — 한글 윈도 인코딩 이름
② `str` — 모든 열을 글자로
③ `rename` — 열 이름 바꾸기
④ `to_datetime` — 글자를 날짜로
⑤ `coerce` — 못 바꾸면 NaN 으로
⑥ `return_station` — 반납 대여소 열
⑦ `~quick` — 조건을 뒤집어 남길 행 고르기
⑧ `hour` — 시각에서 시만 꺼내는 함수
⑨ `strftime(rent_at, '%m-%d')` — 날짜마다 따로 순위
⑩ `1` — 1등만
⑪ `rent_station` — 대여소마다
⑫ `to_parquet` — Parquet 으로 저장